# 05c — Hypothesis testing: City vs Resort

Cùng bộ giả thuyết H1–H4 của notebook **05**, chạy **riêng từng hotel** rồi so sánh effect size.

> Notebook `05b_hypothesis_visualization.ipynb` vẫn là bản hình tổng hợp (gộp portfolio). File này là bản **tách City / Resort**.

| Giả thuyết | Test | Effect size |
|---|---|---|
| H1 `lead_time` | Mann–Whitney U | rank-biserial *r* |
| H1b `lead_time_bin` | Chi-squared | Cramér's V |
| H2 `deposit_type` | Chi-squared | Cramér's V |
| H3 `market_segment` | Chi-squared | Cramér's V |
| H4 3 biến | Logistic + LR test | OR, Pseudo R² |


In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats

%matplotlib inline
sns.set_theme(style="whitegrid", palette="Set2")
plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["axes.titlesize"] = 13
plt.rcParams["axes.labelsize"] = 11

NOTEBOOK_DIR = Path(os.environ.get("VSCODE_NOTEBOOK_DIR", Path.cwd()))
ROOT = NOTEBOOK_DIR.parent if (NOTEBOOK_DIR.parent / "data").is_dir() else NOTEBOOK_DIR
DATA_PATH = ROOT / "data" / "hotel_bookings_v5.csv"
FIG_DIR = ROOT / "reports" / "figures" / "05c"
FIG_DIR.mkdir(parents=True, exist_ok=True)

HOTELS = ["City Hotel", "Resort Hotel"]
HOTEL_COLORS = {"City Hotel": "#4C72B0", "Resort Hotel": "#55A868"}
MONTH_ORDER = [
    "January", "February", "March", "April", "May", "June",
    "July", "August", "September", "October", "November", "December",
]
DAY_ORDER = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
BIN_LABELS = ["0-30", "31-60", "61-90", "91-180", ">180"]
DEPOSIT_ORDER = ["No Deposit", "Non Refund", "Refundable"]
ALPHA = 0.05

print(f"ROOT: {ROOT}")
print(f"DATA: {DATA_PATH}")
print(f"FIG_DIR: {FIG_DIR}")

import statsmodels.api as sm


In [ ]:
def fmt_int(n) -> str:
    return f"{int(round(n)):,}".replace(",", ".")

def fmt_pct(x: float, d: int = 1) -> str:
    return f"{x * 100:.{d}f}%".replace(".", ",")

def fmt_eur(x: float, d: int = 2) -> str:
    s = f"{x:,.{d}f}".replace(",", "X").replace(".", ",").replace("X", ".")
    return f"{s} €"

def savefig(name: str) -> Path:
    path = FIG_DIR / name
    plt.tight_layout()
    plt.savefig(path, dpi=140, bbox_inches="tight")
    print(f"Saved: {path.relative_to(ROOT)}")
    return path

def add_lead_bin(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    edges = [0, 30, 60, 90, 180, float(out["lead_time"].max()) + 1]
    out["lead_time_bin"] = pd.cut(
        out["lead_time"], bins=edges, labels=BIN_LABELS, right=True, include_lowest=True
    )
    return out


def cramers_v(contingency: pd.DataFrame) -> float:
    chi2 = stats.chi2_contingency(contingency)[0]
    n = contingency.to_numpy().sum()
    r, k = contingency.shape
    return float(np.sqrt(chi2 / (n * min(r - 1, k - 1))))

def rank_biserial_from_u(u_stat: float, n0: int, n1: int) -> float:
    return 1 - (2 * u_stat) / (n0 * n1)

def bootstrap_median_diff(group_a, group_b, n_boot: int = 3000, seed: int = 42) -> dict:
    rng = np.random.default_rng(seed)
    a = np.asarray(group_a); b = np.asarray(group_b)
    diffs = np.empty(n_boot)
    for i in range(n_boot):
        diffs[i] = np.median(rng.choice(b, size=len(b), replace=True)) - np.median(rng.choice(a, size=len(a), replace=True))
    lo, hi = np.percentile(diffs, [2.5, 97.5])
    return {"median_diff": float(np.median(diffs)), "ci_low": float(lo), "ci_high": float(hi)}


In [ ]:
df = pd.read_csv(DATA_PATH, usecols=["hotel","lead_time","is_canceled","deposit_type","market_segment"])
df["hotel"] = pd.Categorical(df["hotel"], categories=HOTELS, ordered=True)
df = add_lead_bin(df)
print(f"n={fmt_int(len(df))} | cancel={fmt_pct(df['is_canceled'].mean())}")
for h in HOTELS:
    g = df[df["hotel"]==h]
    print(f"  {h}: n={fmt_int(len(g))} | cancel={fmt_pct(g['is_canceled'].mean())}")


## H1 — lead_time (Mann–Whitney U)


In [ ]:
h1 = {}
for h in HOTELS:
    g = df[df["hotel"]==h]
    a = g.loc[g["is_canceled"]==0, "lead_time"].to_numpy()
    b = g.loc[g["is_canceled"]==1, "lead_time"].to_numpy()
    u, p = stats.mannwhitneyu(a, b, alternative="two-sided")
    r = rank_biserial_from_u(u, len(a), len(b))
    boot = bootstrap_median_diff(a, b)
    h1[h] = dict(u=u, p=p, r=r, med0=np.median(a), med1=np.median(b), mean0=a.mean(), mean1=b.mean(), boot=boot, n0=len(a), n1=len(b))
    print(f"\n{h}: U={u:,.0f} p={p:.2e} r={r:.3f} | median stay {np.median(a):.0f} vs cancel {np.median(b):.0f}")
    print(f"  bootstrap median diff CI [{boot['ci_low']:.1f}, {boot['ci_high']:.1f}]")
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.4))
    sns.boxplot(data=g, x="is_canceled", y="lead_time", showfliers=False, ax=axes[0], color=HOTEL_COLORS[h])
    axes[0].set_xticklabels(["Stay","Canceled"]); axes[0].set_title(h)
    axes[1].axis("off")
    axes[1].text(0.05, 0.9, f"r={r:.3f}\nmedian {np.median(a):.0f} vs {np.median(b):.0f}", va="top")
    savefig(f"h1_{h.split()[0].lower()}.png"); plt.show()


## H1b / H2 / H3 — Chi-squared


In [ ]:
def chi2_report(g, col):
    ct = pd.crosstab(g[col], g["is_canceled"])
    chi2, p, dof, exp = stats.chi2_contingency(ct)
    rates = g.groupby(col, observed=True)["is_canceled"].agg(bookings="count", cancel_rate="mean").reset_index()
    return dict(chi2=chi2, p=p, dof=dof, v=cramers_v(ct), min_exp=float(np.min(exp)), rates=rates)

results = {h: {} for h in HOTELS}
for h in HOTELS:
    g = df[df["hotel"]==h]
    for key, col in [("H1b","lead_time_bin"), ("H2","deposit_type"), ("H3","market_segment")]:
        res = chi2_report(g, col)
        results[h][key] = res
        print(f"{h} {key}: chi2={res['chi2']:.1f} df={res['dof']} p={res['p']:.2e} V={res['v']:.3f}")
        display(res["rates"].round(3))
        fig, ax = plt.subplots(figsize=(10, 4.4))
        if col == "market_segment":
            t = res["rates"].sort_values("cancel_rate", ascending=False)
            sns.barplot(data=t, y=col, x="cancel_rate", ax=ax, color=HOTEL_COLORS[h])
            ax.xaxis.set_major_formatter(lambda x, _: f"{x:.0%}")
        else:
            sns.barplot(data=res["rates"], x=col, y="cancel_rate", ax=ax, color=HOTEL_COLORS[h])
            ax.yaxis.set_major_formatter(lambda x, _: f"{x:.0%}")
            ax.tick_params(axis="x", rotation=15)
        ax.set_title(f"{h} — {col} (V={res['v']:.3f})")
        savefig(f"{key.lower()}_{h.split()[0].lower()}.png"); plt.show()


## H4 — Logistic regression đa biến


In [ ]:
h4 = {}
for h in HOTELS:
    g = df[df["hotel"]==h]
    model_df = g[["is_canceled","lead_time","deposit_type","market_segment"]].copy()
    keep = model_df["market_segment"].value_counts()
    model_df = model_df[model_df["market_segment"].isin(keep[keep>=5].index)]
    model_df["deposit_type"] = pd.Categorical(
        model_df["deposit_type"], categories=["No Deposit", "Non Refund", "Refundable"]
    )
    seg_cats = ["Direct"] + sorted(c for c in model_df["market_segment"].unique() if c != "Direct")
    model_df["market_segment"] = pd.Categorical(model_df["market_segment"], categories=seg_cats)
    X_cat = pd.get_dummies(model_df[["deposit_type","market_segment"]], drop_first=True)
    X = sm.add_constant(pd.concat([model_df[["lead_time"]], X_cat], axis=1).astype(float))
    y = model_df["is_canceled"].astype(float)
    logit = sm.Logit(y, X).fit(disp=False, maxiter=500, method="lbfgs")
    null = sm.Logit(y, np.ones((len(y),1))).fit(disp=False, maxiter=500, method="lbfgs")
    lr = 2*(logit.llf - null.llf)
    lr_df = len(logit.params)-1
    lr_p = stats.chi2.sf(lr, lr_df)
    or_30 = np.exp(logit.params["lead_time"]*30)
    h4[h] = dict(n=int(logit.nobs), r2=float(logit.prsquared), lr=float(lr), lr_p=float(lr_p), or30=float(or_30), params=logit.params)
    print(f"\n{h}: n={logit.nobs:,}  pseudo-R2={logit.prsquared:.3f}  LR p={lr_p:.2e}  OR+30d={or_30:.3f}")
    ors = np.exp(logit.params.drop("const")).sort_values()
    fig, ax = plt.subplots(figsize=(10, 5.5))
    ax.barh(ors.index.astype(str), ors.values, color=HOTEL_COLORS[h])
    ax.axvline(1, color="black", lw=0.8)
    ax.set_title(f"{h} — Odds ratio")
    savefig(f"h4_{h.split()[0].lower()}.png"); plt.show()


## Dashboard effect size


In [ ]:
rows = []
for h in HOTELS:
    rows += [
        {"hotel": h, "test": "H1 |r|", "value": abs(h1[h]["r"])},
        {"hotel": h, "test": "H1b V", "value": results[h]["H1b"]["v"]},
        {"hotel": h, "test": "H2 V", "value": results[h]["H2"]["v"]},
        {"hotel": h, "test": "H3 V", "value": results[h]["H3"]["v"]},
        {"hotel": h, "test": "H4 pseudo-R2", "value": h4[h]["r2"]},
    ]
eff = pd.DataFrame(rows)
display(eff.pivot(index="test", columns="hotel", values="value").round(3))
fig, ax = plt.subplots(figsize=(11,5))
sns.barplot(data=eff, x="test", y="value", hue="hotel", hue_order=HOTELS, palette=HOTEL_COLORS, ax=ax)
ax.set_title("Effect size City vs Resort")
savefig("07_effect_size_compare.png"); plt.show()
